# 🌫️ India AQI Data Analysis
**Author:** Ankit Gupta  
**Project:** Air Quality Index (AQI) Analysis of India (2015–2020)

---

## What is this project about?
This notebook analyzes air pollution data across **26 cities** and **230 monitoring stations** in India.  
We use 5 CSV files provided by CPCB (Central Pollution Control Board) to:
1. Load and understand the data
2. Clean all messy/missing values properly
3. Join the tables together for richer analysis
4. Extract meaningful insights about India's air quality

## 📦 Dataset Source
**Kaggle:** [Air Quality Data in India — Rohan Rao](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india?resource=download)  
Download all 5 CSV files and place them inside the `dataset/` folder before running this notebook.

## Dataset Files
| File | What it contains |
|------|------------------|
| `city_day.csv` | Daily pollution readings for each city |
| `city_hour.csv` | Hourly pollution readings for each city |
| `stations.csv` | Info about each monitoring station (name, city, state) |
| `station_day.csv` | Daily readings per individual monitoring station |
| `station_hour.csv` | Hourly readings per individual monitoring station |

---
## Step 1: Import Libraries

We need two main Python libraries:
- **pandas** — for working with tables (loading CSVs, cleaning, joining, grouping)
- **numpy** — for math operations like calculating medians, handling missing values (`NaN`)

In [ ]:
# pandas  → the main library for working with data tables
# numpy   → for math/number operations
# warnings → to suppress non-critical warning messages for cleaner output

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')  # hide harmless warnings

# Show all columns when printing a table (don't truncate)
pd.set_option('display.max_columns', None)

# Show numbers with 2 decimal places
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries imported successfully.')

---
## Step 2: Load the CSV Files

We read all 5 CSV files from the `dataset/` folder.  
`parse_dates` tells pandas to treat the date/time columns as actual dates (not plain text),  
so we can later filter by year, month, etc.

In [ ]:
# Read city-level data
city_day   = pd.read_csv('dataset/city_day.csv',  parse_dates=['Date'])      # 1 row = 1 city × 1 day
city_hour  = pd.read_csv('dataset/city_hour.csv', parse_dates=['Datetime'])  # 1 row = 1 city × 1 hour

# Read station metadata (name, city, state)
stations   = pd.read_csv('dataset/stations.csv')

# Read station-level measurement data
# low_memory=False → prevents a warning about mixed data types in a column
station_day  = pd.read_csv('dataset/station_day.csv',  parse_dates=['Date'])
station_hour = pd.read_csv('dataset/station_hour.csv', parse_dates=['Datetime'], low_memory=False)

print('✅ All 5 datasets loaded successfully!\n')

# Show a quick summary of each file: name, number of rows and columns
for name, df in [('city_day', city_day), ('city_hour', city_hour), ('stations', stations),
                 ('station_day', station_day), ('station_hour', station_hour)]:
    print(f'  {name:20s} → {df.shape[0]:>9,} rows  ×  {df.shape[1]} columns')

---
## Step 3: Data Cleaning

Raw data is rarely perfect. Here is a summary of problems we found and how we fix them:

| Problem | Where it occurs | How we fix it |
|---|---|---|
| Missing pollutant values (6–80%) | All measurement files | Interpolation → fill → median |
| Missing AQI (15–22%) | All measurement files | Recalculate using CPCB formula |
| Inconsistent AQI_Bucket labels | All measurement files | Re-derive from AQI using official CPCB ranges |
| AQI > 500 (impossible — India scale max is 500) | city_day, station_day | Cap at 500, set to NaN first |
| Extreme outliers in CO, Benzene, Toluene | city_day, station_day | Clip at 99th percentile |
| Missing Status in stations (42%) | stations.csv | Fill with 'Unknown' |

### 3.1 Helper: Before vs After Missing Values Report

This function prints a table showing how many values were missing **before** and **after** cleaning,  
so we can verify that our cleaning actually worked.

In [ ]:
def cleaning_report(before_df, after_df, name):
    """
    Prints a side-by-side comparison of missing values
    before and after cleaning for a given dataset.
    """
    # Find columns that had any missing values
    cols_with_missing = [
        col for col in before_df.columns
        if before_df[col].isnull().any() or after_df[col].isnull().any()
    ]

    if not cols_with_missing:
        print(f'[{name}] → No missing values found. Nothing to report.')
        return

    # Calculate % missing before and after
    before_pct = (before_df[cols_with_missing].isnull().sum() / len(before_df) * 100).round(2)
    after_pct  = (after_df[cols_with_missing].isnull().sum()  / len(after_df)  * 100).round(2)

    report = pd.DataFrame({
        'Missing Before (%)': before_pct,
        'Missing After (%)' : after_pct,
        'Improvement (%)'   : (before_pct - after_pct).round(2)
    })
    # Only show rows where there was a missing value before cleaning
    report = report[report['Missing Before (%)'] > 0].sort_values('Missing Before (%)', ascending=False)

    print(f'\n── Cleaning Report: {name} ──────────────────────')
    display(report)

print('✅ Helper function defined.')

### 3.2 CPCB AQI Formula

India uses the **CPCB AQI standard**. Here is how it works:
- For each pollutant (PM2.5, PM10, NO2, etc.), compute a **sub-index** using official concentration breakpoints
- The **final AQI = the highest sub-index** among all available pollutants (minimum 3 sub-indices required)
- The **AQI Bucket** (Good / Satisfactory / Moderate / Poor / Very Poor / Severe) is assigned based on the numeric AQI

We use this to **recalculate missing AQI values** from scratch instead of leaving them empty.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CPCB Breakpoints for each pollutant
# Format: (concentration_low, concentration_high, AQI_low, AQI_high)
# Source: CPCB National Air Quality Index (2014)
# ─────────────────────────────────────────────────────────────
CPCB_BREAKPOINTS = {
    'PM2.5'  : [(0,30,0,50),(30,60,51,100),(60,90,101,200),(90,120,201,300),(120,250,301,400),(250,380,401,500)],
    'PM10'   : [(0,50,0,50),(50,100,51,100),(100,250,101,200),(250,350,201,300),(350,430,301,400),(430,580,401,500)],
    'NO2'    : [(0,40,0,50),(40,80,51,100),(80,180,101,200),(180,280,201,300),(280,400,301,400),(400,800,401,500)],
    'NH3'    : [(0,200,0,50),(200,400,51,100),(400,800,101,200),(800,1200,201,300),(1200,1800,301,400),(1800,2400,401,500)],
    'SO2'    : [(0,40,0,50),(40,80,51,100),(80,380,101,200),(380,800,201,300),(800,1600,301,400),(1600,2100,401,500)],
    'CO'     : [(0,1,0,50),(1,2,51,100),(2,10,101,200),(10,17,201,300),(17,34,301,400),(34,46,401,500)],
    'O3'     : [(0,50,0,50),(50,100,51,100),(100,168,101,200),(168,208,201,300),(208,748,301,400),(748,1000,401,500)],
    'Benzene': [(0,5,0,50),(5,10,51,100),(10,40,101,200),(40,100,201,300),(100,1000,301,400),(1000,2000,401,500)],
}

# Official AQI category order (from cleanest to most polluted)
AQI_BUCKET_ORDER = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']


def compute_sub_index(concentration, breakpoints):
    """
    Calculate the AQI sub-index for one pollutant.
    Uses linear interpolation between the matching concentration breakpoints.
    Returns NaN if the value is missing or negative.
    """
    if pd.isna(concentration) or concentration < 0:
        return np.nan
    for (c_lo, c_hi, i_lo, i_hi) in breakpoints:
        if c_lo <= concentration <= c_hi:
            # Linear interpolation formula
            return i_lo + (concentration - c_lo) * (i_hi - i_lo) / (c_hi - c_lo)
    return 500.0  # beyond the highest breakpoint → capped at 500


def calculate_aqi(row):
    """
    Calculate the overall AQI for one row (one measurement record).
    AQI = max of all available sub-indices.
    Requires at least 3 valid sub-indices; otherwise returns NaN.
    """
    sub_indices = []
    for pollutant, breakpoints in CPCB_BREAKPOINTS.items():
        if pollutant in row.index:
            si = compute_sub_index(row[pollutant], breakpoints)
            if not pd.isna(si):
                sub_indices.append(si)
    if len(sub_indices) >= 3:
        return round(max(sub_indices), 0)  # AQI = highest sub-index
    return np.nan  # not enough data to compute AQI


def aqi_to_bucket(aqi):
    """
    Convert a numeric AQI value to its official CPCB category label.
    """
    if pd.isna(aqi): return np.nan
    if aqi <= 50:    return 'Good'
    if aqi <= 100:   return 'Satisfactory'
    if aqi <= 200:   return 'Moderate'
    if aqi <= 300:   return 'Poor'
    if aqi <= 400:   return 'Very Poor'
    return 'Severe'

print('✅ CPCB AQI formula functions defined.')

### 3.3 Physical Upper Limits for Each Pollutant

Some recorded values are clearly instrument errors (e.g., AQI = 2049 is impossible on India's 0–500 scale).  
We define a maximum physically plausible value for each pollutant.  
Any value above the cap is replaced with `NaN` (treated as missing) and then re-filled by our imputation pipeline.

In [ ]:
# Maximum valid values for each pollutant (based on CPCB instrument range & scientific literature)
# Values above these are considered errors and set to NaN
PHYSICAL_CAPS = {
    'PM2.5'  : 900,   # µg/m³ — extreme wildfire smoke upper bound
    'PM10'   : 1500,  # µg/m³ — CPCB instrument range
    'NO'     : 1000,  # µg/m³
    'NO2'    : 1000,  # µg/m³
    'NOx'    : 2000,  # µg/m³
    'NH3'    : 400,   # µg/m³
    'CO'     : 50,    # mg/m³  — extreme industrial upper bound
    'SO2'    : 1000,  # µg/m³
    'O3'     : 1000,  # µg/m³
    'Benzene': 100,   # µg/m³
    'Toluene': 200,   # µg/m³
    'Xylene' : 200,   # µg/m³
    'AQI'    : 500,   # India CPCB scale maximum
}

# List of all pollutant column names (used throughout cleaning)
POLLUTANT_COLS = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
                  'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']

print('✅ Physical caps and pollutant column list defined.')

### 3.4 The Main Cleaning Function

This single function runs **all 9 cleaning steps** on any measurement dataset (city_day, city_hour, station_day, station_hour).  
We just call it with different parameters for each file.

**The 9 steps explained simply:**
1. Remove duplicate rows (same city/station + same time)
2. Remove rows where every single pollutant is blank (useless records)
3. Replace impossible values with blank (`NaN`) using our physical caps
4. Clip extreme values: cap anything above the 99th percentile of its own city/station group
5. Fill gaps using **linear interpolation** (estimate based on neighbours in the time series)
6. Fill remaining gaps using **forward/backward fill** (carry the last known value forward)
7. Fill any still-remaining gaps with the **city/station's own median** (long-run average)
8. **Recalculate AQI** where it was missing or wrong, using the official CPCB formula
9. **Rederive AQI_Bucket** from the clean AQI value using official CPCB category breakpoints

In [ ]:
def clean_dataset(df, group_col, time_col, is_hourly=False, name=''):
    """
    Cleans a measurement DataFrame through 9 steps.

    Parameters
    ----------
    df         : the raw DataFrame to clean
    group_col  : column to group by ('City' for city data, 'StationId' for station data)
    time_col   : the date/time column ('Date' for daily, 'Datetime' for hourly)
    is_hourly  : True for hourly data (shorter interpolation limits), False for daily
    name       : label used in the printed progress message

    Returns
    -------
    Cleaned DataFrame
    """
    df = df.copy()  # always work on a copy so the original is not modified

    # Interpolation limits: don't fill more than 3 consecutive hours or 7 consecutive days
    interp_limit = 3 if is_hourly else 7
    ffill_limit  = 6 if is_hourly else 14

    # ── Step 1: Remove exact duplicate rows ──────────────────────────────
    original_count = len(df)
    df.drop_duplicates(subset=[group_col, time_col], keep='first', inplace=True)
    dupes_removed = original_count - len(df)

    # ── Step 2: Remove rows where ALL pollutants are missing ──────────────
    poll_cols = [c for c in POLLUTANT_COLS if c in df.columns]  # only use cols that exist
    all_blank = df[poll_cols].isnull().all(axis=1)              # True if every pollutant is NaN
    df = df[~all_blank].copy()                                  # keep only rows that have some data
    empty_removed = all_blank.sum()

    # ── Step 3: Replace physically impossible values with NaN ─────────────
    for col, cap in PHYSICAL_CAPS.items():
        if col in df.columns:
            df.loc[df[col] > cap, col] = np.nan  # set to NaN — will be filled later

    # ── Step 4: Clip outliers at each group's 99th percentile ────────────
    # This prevents one extremely high value from distorting city/station averages
    for col in poll_cols:
        p99 = df.groupby(group_col)[col].transform(lambda x: x.quantile(0.99))
        df[col] = df[col].clip(upper=p99)

    # ── Steps 5 & 6: Interpolate and fill gaps within each group ─────────
    # Sort by (group, time) so interpolation works along the time axis
    df.sort_values([group_col, time_col], inplace=True)

    def fill_gaps_in_group(group):
        """Apply interpolation + forward/back fill to one city/station group."""
        for col in poll_cols:
            # Step 5: linear interpolation — estimates missing values from surrounding points
            group[col] = group[col].interpolate(
                method='linear', limit=interp_limit, limit_direction='both'
            )
            # Step 6: carry last known value forward, then backward
            group[col] = group[col].ffill(limit=ffill_limit).bfill(limit=ffill_limit)
        return group

    df = df.groupby(group_col, group_keys=False).apply(fill_gaps_in_group)

    # ── Step 7: Fill any still-missing values with the group's own median ─
    for col in poll_cols:
        group_median  = df.groupby(group_col)[col].transform('median')  # city/station median
        df[col] = df[col].fillna(group_median)         # fill with city/station median
        df[col] = df[col].fillna(df[col].median())     # final fallback: overall dataset median

    # ── Step 8: Recalculate AQI where it is missing or was > 500 ─────────
    needs_aqi = df['AQI'].isna() | (df['AQI'] > 500)
    if needs_aqi.any():
        df.loc[needs_aqi, 'AQI'] = df[needs_aqi].apply(calculate_aqi, axis=1)

    # Fill any remaining AQI gaps with group median, then overall median
    group_aqi_median = df.groupby(group_col)['AQI'].transform('median')
    df['AQI'] = df['AQI'].fillna(group_aqi_median)
    df['AQI'] = df['AQI'].fillna(df['AQI'].median())
    df['AQI'] = df['AQI'].clip(upper=500).round(0)  # cap at 500 and round to whole number

    # ── Step 9: Rederive AQI_Bucket from the cleaned AQI ─────────────────
    # Overwrite the entire column so labels are consistent with our cleaned AQI values
    df['AQI_Bucket'] = df['AQI'].apply(aqi_to_bucket)
    df['AQI_Bucket'] = pd.Categorical(df['AQI_Bucket'],
                                       categories=AQI_BUCKET_ORDER, ordered=True)

    # Final sort and reset row numbers
    df.sort_values([group_col, time_col], inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f'✅ [{name}] Cleaned — duplicates removed: {dupes_removed}, '
          f'empty rows dropped: {empty_removed}, final shape: {df.shape}')
    return df

print('✅ Cleaning function defined.')

### 3.5 Clean `city_day`
Group by **City**, time column is **Date**, daily data.

In [ ]:
city_day_clean = clean_dataset(
    df       = city_day,
    group_col= 'City',
    time_col = 'Date',
    is_hourly= False,
    name     = 'city_day'
)

# Show how many missing values we fixed
cleaning_report(city_day, city_day_clean, 'city_day')

### 3.6 Clean `city_hour`
Group by **City**, time column is **Datetime**, hourly data.

In [ ]:
city_hour_clean = clean_dataset(
    df       = city_hour,
    group_col= 'City',
    time_col = 'Datetime',
    is_hourly= True,
    name     = 'city_hour'
)

cleaning_report(city_hour, city_hour_clean, 'city_hour')

### 3.7 Clean `station_day`
Group by **StationId**, time column is **Date**, daily data.

In [ ]:
station_day_clean = clean_dataset(
    df       = station_day,
    group_col= 'StationId',
    time_col = 'Date',
    is_hourly= False,
    name     = 'station_day'
)

cleaning_report(station_day, station_day_clean, 'station_day')

### 3.8 Clean `station_hour`
Group by **StationId**, time column is **Datetime**, hourly data.

In [ ]:
station_hour_clean = clean_dataset(
    df       = station_hour,
    group_col= 'StationId',
    time_col = 'Datetime',
    is_hourly= True,
    name     = 'station_hour'
)

cleaning_report(station_hour, station_hour_clean, 'station_hour')

### 3.9 Clean `stations` (metadata)
This file has no pollutant readings — just station info.  
We only need to strip extra spaces and fill the missing `Status` column.

In [ ]:
stations_clean = stations.copy()

# Remove leading/trailing spaces from all text columns
for col in stations_clean.select_dtypes(include='object').columns:
    stations_clean[col] = stations_clean[col].str.strip()

# Fill missing Status values with 'Unknown'
stations_clean['Status'] = stations_clean['Status'].fillna('Unknown')

# Remove any duplicate stations
stations_clean.drop_duplicates(subset=['StationId'], keep='first', inplace=True)

print(f'✅ stations_clean: {stations_clean.shape[0]} stations, {stations_clean.isnull().sum().sum()} missing values')
display(stations_clean.head())

### 3.10 Cleaning Summary
A final overview of how many rows and missing cells were fixed across all 5 files.

In [ ]:
# Build a summary table: raw vs clean for each dataset
summary = []
for label, raw, clean in [
    ('city_day',     city_day,     city_day_clean),
    ('city_hour',    city_hour,    city_hour_clean),
    ('station_day',  station_day,  station_day_clean),
    ('station_hour', station_hour, station_hour_clean),
    ('stations',     stations,     stations_clean),
]:
    raw_missing   = raw.isnull().sum().sum()
    clean_missing = clean.isnull().sum().sum()
    summary.append({
        'Dataset'             : label,
        'Raw Rows'            : raw.shape[0],
        'Clean Rows'          : clean.shape[0],
        'Missing Before'      : raw_missing,
        'Missing After'       : clean_missing,
        'Values Fixed'        : raw_missing - clean_missing,
    })

print('=== Overall Cleaning Summary ===')
display(pd.DataFrame(summary).set_index('Dataset'))

---
## Step 4: Join the Tables

Now we combine the cleaned files into **master tables** for easier analysis.

| Join | How | Result |
|---|---|---|
| `station_day_clean` + `stations_clean` | LEFT JOIN on `StationId` | Adds city, state, station name to each measurement |
| `station_hour_clean` + `stations_clean` | LEFT JOIN on `StationId` | Same, but for hourly data |
| `city_day_clean` + station_day_master | UNION (stack) | One big daily table with a `Source` column |
| `city_hour_clean` + station_hour_master | UNION (stack) | One big hourly table |

### 4.1 Station-Day Master Table
Attach station metadata (City, State, StationName) to daily station readings.

In [ ]:
# LEFT JOIN: keep all rows from station_day_clean,
# and add matching columns from stations_clean based on StationId
station_day_master = station_day_clean.merge(
    stations_clean[['StationId', 'StationName', 'City', 'State', 'Status']],
    on  = 'StationId',
    how = 'left'        # keep all rows from left table even if no match found
)

print(f'✅ station_day_master: {station_day_master.shape[0]:,} rows × {station_day_master.shape[1]} columns')
display(station_day_master.head(3))

### 4.2 Station-Hour Master Table

In [ ]:
station_hour_master = station_hour_clean.merge(
    stations_clean[['StationId', 'StationName', 'City', 'State', 'Status']],
    on  = 'StationId',
    how = 'left'
)

print(f'✅ station_hour_master: {station_hour_master.shape[0]:,} rows × {station_hour_master.shape[1]} columns')

### 4.3 Unified Daily Master Table
Stack city-level and station-level daily data into one table.  
A `Source` column tells us whether each row came from city data or station data.

In [ ]:
# All measurement columns we want to keep in the combined table
ALL_MEASUREMENT_COLS = POLLUTANT_COLS + ['AQI', 'AQI_Bucket']

# Prepare the city_day slice — add empty station columns so both tables have the same structure
city_day_slice = city_day_clean[['City', 'Date'] + ALL_MEASUREMENT_COLS].copy()
city_day_slice['Source']      = 'city'   # tag so we know where this row came from
city_day_slice['StationId']   = np.nan
city_day_slice['StationName'] = np.nan
city_day_slice['State']       = np.nan

# Prepare the station_day slice — already has station columns from the join above
station_day_slice = station_day_master[['City', 'Date', 'StationId', 'StationName', 'State']
                                        + ALL_MEASUREMENT_COLS].copy()
station_day_slice['Source'] = 'station'

# Stack the two tables on top of each other (they have the same columns)
daily_master = pd.concat([city_day_slice, station_day_slice], ignore_index=True)
daily_master.sort_values(['City', 'Date'], inplace=True)
daily_master.reset_index(drop=True, inplace=True)

print(f'✅ daily_master: {daily_master.shape[0]:,} rows × {daily_master.shape[1]} columns')
print(f'   Date range : {daily_master["Date"].min().date()} → {daily_master["Date"].max().date()}')
print(f'   Cities     : {daily_master["City"].nunique()}')

### 4.4 Unified Hourly Master Table

In [ ]:
city_hour_slice = city_hour_clean[['City', 'Datetime'] + ALL_MEASUREMENT_COLS].copy()
city_hour_slice['Source']      = 'city'
city_hour_slice['StationId']   = np.nan
city_hour_slice['StationName'] = np.nan
city_hour_slice['State']       = np.nan

station_hour_slice = station_hour_master[['City', 'Datetime', 'StationId', 'StationName', 'State']
                                          + ALL_MEASUREMENT_COLS].copy()
station_hour_slice['Source'] = 'station'

hourly_master = pd.concat([city_hour_slice, station_hour_slice], ignore_index=True)
hourly_master.sort_values(['City', 'Datetime'], inplace=True)
hourly_master.reset_index(drop=True, inplace=True)

print(f'✅ hourly_master: {hourly_master.shape[0]:,} rows × {hourly_master.shape[1]} columns')

---
## Step 5: ⭐ Top 5 Key Insights

> The five most important findings from this dataset.

| # | Insight | Key Number |
|---|---|---|
| 1 | India's AQI improved **47%** from 2015 to 2020 | 212.5 → 113.5 |
| 2 | **PM10** is the strongest predictor of AQI (r = 0.80) | Correlation 0.80 |
| 3 | **Winter** is 2× more polluted than Monsoon season | Winter avg AQI 220 |
| 4 | **Delhi, Patna & Gurugram** are most polluted; **Aizawl** cleanest | Delhi avg AQI 259 |
| 5 | **COVID lockdown** cut AQI by **42%** in Q2 2020 | 145 → 83 |

### ⭐ Insight 1 — India's AQI Improved 47% from 2015 to 2020

The national average AQI fell from **212.5** (Poor category) in 2015 to **113.5** (Moderate) in 2020.  
This is a 47% improvement — though part of the 2020 drop is due to COVID-19 lockdowns.

In [ ]:
# Extract year from the Date column
city_day_clean['Year'] = city_day_clean['Date'].dt.year

# Calculate the average AQI for each year across all cities
yearly_aqi = (
    city_day_clean.groupby('Year')['AQI']
    .mean()                                     # average AQI per year
    .reset_index()
    .rename(columns={'AQI': 'Avg_AQI'})
)
yearly_aqi['Avg_AQI']      = yearly_aqi['Avg_AQI'].round(1)
yearly_aqi['AQI_Category'] = yearly_aqi['Avg_AQI'].apply(aqi_to_bucket)  # add label

# Calculate overall % change from first year to last year
first_year_aqi = yearly_aqi['Avg_AQI'].iloc[0]
last_year_aqi  = yearly_aqi['Avg_AQI'].iloc[-1]
pct_change     = (last_year_aqi - first_year_aqi) / first_year_aqi * 100

print('Year-on-year national average AQI (all 26 cities):')
display(yearly_aqi)
print(f'\n📌 Overall change 2015 → 2020: {pct_change:.1f}%')

### ⭐ Insight 2 — PM10 is the #1 AQI Driver (r = 0.80)

We calculate the **Pearson correlation** between each pollutant and AQI.  
A value close to **+1** means a strong positive relationship (higher pollutant → higher AQI).  

**PM10** (coarse dust particles from roads, construction) has the highest correlation at **r = 0.80**,  
followed by CO (0.68) and PM2.5 (0.66).

In [ ]:
# Columns to include in the correlation analysis
numeric_cols = POLLUTANT_COLS + ['AQI']

# Compute correlations of each pollutant with AQI and sort highest to lowest
corr_with_aqi = (
    city_day_clean[numeric_cols]
    .corr()['AQI']         # correlation of every column with the AQI column
    .drop('AQI')           # remove the AQI-vs-AQI row (always 1.0)
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'index': 'Pollutant', 'AQI': 'Correlation_with_AQI'})
)
corr_with_aqi['Correlation_with_AQI'] = corr_with_aqi['Correlation_with_AQI'].round(3)

print('📌 Pollutant correlation with AQI (highest = biggest influence):')
display(corr_with_aqi)

### ⭐ Insight 3 — Winter & Post-Monsoon Are the Most Polluted Seasons

We classify each month into one of India's four meteorological seasons and compare AQI.  
**Winter average AQI = 220 (Poor)** — nearly double the **Monsoon average of 115 (Moderate)**.  
November PM2.5 reaches **110.5 µg/m³** — that is 3× the CPCB annual safe limit of 40 µg/m³.

In [ ]:
# Add month number to the dataframe
city_day_clean['Month'] = city_day_clean['Date'].dt.month

# Map each month to its season
def get_season(month):
    if month in [12, 1, 2]:   return 'Winter (Dec–Feb)'
    if month in [3, 4, 5]:    return 'Pre-Monsoon (Mar–May)'
    if month in [6, 7, 8, 9]: return 'Monsoon (Jun–Sep)'
    return 'Post-Monsoon (Oct–Nov)'

city_day_clean['Season'] = city_day_clean['Month'].apply(get_season)

# Calculate average, median, and maximum AQI + PM levels for each season
season_stats = (
    city_day_clean
    .groupby('Season')[['AQI', 'PM2.5', 'PM10']]
    .agg(['mean', 'median', 'max'])
    .round(1)
)
# Show worst season first
season_order = ['Winter (Dec–Feb)', 'Post-Monsoon (Oct–Nov)', 'Pre-Monsoon (Mar–May)', 'Monsoon (Jun–Sep)']
season_stats = season_stats.reindex(season_order)

print('📌 AQI and PM levels by season (worst → best):')
display(season_stats)

# Monthly PM2.5 — shows which months are dangerous
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly_pm25 = city_day_clean.groupby('Month')['PM2.5'].mean().round(1)
monthly_pm25.index = monthly_pm25.index.map(month_names)

print('\n📌 Monthly average PM2.5 (µg/m³) — CPCB safe limit = 40 µg/m³:')
display(monthly_pm25.rename('Avg_PM2.5'))

### ⭐ Insight 4 — Delhi, Patna & Gurugram Most Polluted; Aizawl Cleanest

We rank all 26 cities by their average AQI over the full 5-year period.  
North Indian cities (Indo-Gangetic Plain) dominate the top of the pollution list.  
North-Eastern cities (Aizawl, Shillong) consistently have the cleanest air.

In [ ]:
# Calculate average, median, and max AQI per city over all years
city_ranking = (
    city_day_clean
    .groupby('City')['AQI']
    .agg(Avg_AQI='mean', Median_AQI='median', Max_AQI='max', Total_Days='count')
    .round(1)
    .sort_values('Avg_AQI', ascending=False)  # most polluted first
    .reset_index()
)
city_ranking['Category'] = city_ranking['Avg_AQI'].apply(aqi_to_bucket)  # add category label

print('📌 Top 10 Most Polluted Cities (5-year average):')
display(city_ranking.head(10))

print('\n📌 Top 5 Cleanest Cities:')
display(city_ranking.tail(5).sort_values('Avg_AQI'))

### ⭐ Insight 5 — COVID-19 Lockdown Cut AQI by 42% in Q2 2020

India's national lockdown started on **25 March 2020**.  
We compare Q1 2020 (Jan–Mar, pre-lockdown) vs Q2 2020 (Apr–Jun, lockdown period).  
AQI dropped from **145 → 83** — a 42% improvement in just one quarter.  
This shows how much human activity (transport, industry) contributes to pollution.

In [ ]:
# Filter to year 2020 only
data_2020 = city_day_clean[city_day_clean['Year'] == 2020].copy()

# Add quarter number (1 = Jan–Mar, 2 = Apr–Jun, 3 = Jul–Sep)
data_2020['Quarter'] = data_2020['Date'].dt.quarter

# Give each quarter a readable label
quarter_labels = {
    1: 'Q1  Jan–Mar  (Pre-lockdown)',
    2: 'Q2  Apr–Jun  (Lockdown)',
    3: 'Q3  Jul       (Partial unlock)'
}
data_2020['Quarter_Label'] = data_2020['Quarter'].map(quarter_labels)

# Average AQI and key pollutants per quarter
lockdown_impact = (
    data_2020
    .groupby('Quarter_Label')[['AQI', 'PM2.5', 'PM10', 'NO2', 'CO']]
    .mean()
    .round(1)
)
print('📌 2020 AQI and pollutant levels by quarter:')
display(lockdown_impact)

# Calculate % improvement from Q1 to Q2
q1_aqi = data_2020[data_2020['Quarter'] == 1]['AQI'].mean()
q2_aqi = data_2020[data_2020['Quarter'] == 2]['AQI'].mean()
print(f'\n📌 AQI reduction from Q1 → Q2 2020: {(q1_aqi - q2_aqi) / q1_aqi * 100:.1f}% improvement')

---
## Step 6: Extended Analysis

### 6.1 AQI Bucket Distribution — What Category Is India In Most Often?

In [ ]:
# Count how many city-days fall into each AQI category
bucket_counts = city_day_clean['AQI_Bucket'].value_counts().reset_index()
bucket_counts.columns = ['AQI_Bucket', 'Count']

# Sort in official CPCB order (Good → Severe)
bucket_counts['AQI_Bucket'] = pd.Categorical(
    bucket_counts['AQI_Bucket'], categories=AQI_BUCKET_ORDER, ordered=True
)
bucket_counts.sort_values('AQI_Bucket', inplace=True)

# Calculate percentage share
total = bucket_counts['Count'].sum()
bucket_counts['Share %'] = (bucket_counts['Count'] / total * 100).round(1)

print(f'Total city-day records: {total:,}')
display(bucket_counts)

### 6.2 State-Level AQI Ranking (using Station Data)
Station data tells us **which state** each reading belongs to, so we can rank states.

In [ ]:
# Average AQI per state using the station_day_master table (has State column)
state_ranking = (
    station_day_master
    .groupby('State')['AQI']
    .agg(Avg_AQI='mean', Median_AQI='median', Max_AQI='max')
    .dropna()
    .round(1)
    .sort_values('Avg_AQI', ascending=False)
    .reset_index()
)
state_ranking['Category'] = state_ranking['Avg_AQI'].apply(aqi_to_bucket)

print('📌 AQI ranking by State (worst → best):')
display(state_ranking)

### 6.3 Monthly Pollution Trend — Which Months Are Most Dangerous?

In [ ]:
# Average AQI, PM2.5, PM10, NO2 for each month of the year
monthly_trend = (
    city_day_clean
    .groupby('Month')[['AQI', 'PM2.5', 'PM10', 'NO2']]
    .mean()
    .round(1)
    .reset_index()
)
# Replace month numbers with names for readability
monthly_trend['Month'] = monthly_trend['Month'].map(month_names)
monthly_trend.set_index('Month', inplace=True)

print('📌 Monthly average AQI and key pollutants (all cities combined):')
display(monthly_trend)

### 6.4 Worst Single Pollution Day per City
The single most polluted day ever recorded in each city.

In [ ]:
# For each city, find the row with the maximum AQI value
worst_days = (
    city_day_clean
    .loc[city_day_clean.groupby('City')['AQI'].idxmax()]  # index of max AQI per city
    [['City', 'Date', 'AQI', 'AQI_Bucket', 'PM2.5', 'PM10']]
    .sort_values('AQI', ascending=False)
    .reset_index(drop=True)
)

print('📌 All-time worst pollution day per city:')
display(worst_days)

### 6.5 Which Cities Improved the Most? (2015 vs 2019)
Comparing average AQI in 2015 vs 2019 (we skip 2020 to avoid COVID distortion).

In [ ]:
# Calculate annual average AQI for each city in 2015 and 2019
city_year_aqi = (
    city_day_clean[city_day_clean['Year'].isin([2015, 2019])]
    .groupby(['City', 'Year'])['AQI']
    .mean()
    .unstack()   # pivot: one column per year
    .dropna()    # remove cities that don't have data for both years
    .round(1)
)

# Calculate % change (negative = improvement, positive = got worse)
city_year_aqi['Change %'] = ((city_year_aqi[2019] - city_year_aqi[2015])
                              / city_year_aqi[2015] * 100).round(1)
city_year_aqi.columns = ['AQI_2015', 'AQI_2019', 'Change %']
city_year_aqi.sort_values('Change %', inplace=True)  # most improved first

print('📌 City AQI change from 2015 → 2019  (negative = improvement):')
display(city_year_aqi)

### 6.6 Hourly Pattern — When Is the Air Cleanest During the Day?

In [ ]:
# Extract hour of day from the Datetime column
city_hour_clean['Hour'] = city_hour_clean['Datetime'].dt.hour

# Average AQI and PM2.5 for each hour of the day (0–23)
hourly_pattern = (
    city_hour_clean
    .groupby('Hour')[['AQI', 'PM2.5']]
    .mean()
    .round(1)
    .reset_index()
)

cleanest_hour = hourly_pattern.loc[hourly_pattern['AQI'].idxmin(), 'Hour']
dirtiest_hour = hourly_pattern.loc[hourly_pattern['AQI'].idxmax(), 'Hour']

print(f'📌 Cleanest hour of the day : {cleanest_hour}:00  '
      f'(Avg AQI = {hourly_pattern["AQI"].min():.1f})')
print(f'📌 Most polluted hour       : {dirtiest_hour}:00  '
      f'(Avg AQI = {hourly_pattern["AQI"].max():.1f})')
print()
display(hourly_pattern)

### 6.7 Pollutant Descriptive Statistics
Basic statistics for every pollutant in the cleaned city-day dataset.

In [ ]:
# describe() gives count, mean, std, min, 25%, median, 75%, max for each column
print('📌 Descriptive statistics for all pollutants (city_day_clean):')
display(city_day_clean[numeric_cols].describe().round(2))

### 6.8 Full Pollutant Correlation Matrix
How strongly is each pollutant related to every other pollutant?

In [ ]:
# corr() returns a matrix: row × column = correlation between those two variables
# Values close to +1 = strong positive relationship
# Values close to  0 = no relationship
# Values close to -1 = inverse relationship

corr_matrix = city_day_clean[numeric_cols].corr().round(2)
print('📌 Full pollutant correlation matrix (city_day_clean):')
display(corr_matrix)

### 6.9 How Often Does Each City Have 'Good' Air Quality?
What percentage of days had AQI ≤ 50 (the 'Good' category)?

In [ ]:
# Count the number of 'Good' AQI days per city
good_air_days = (
    city_day_clean[city_day_clean['AQI_Bucket'] == 'Good']
    .groupby('City')
    .size()
    .reset_index(name='Good_Days')
)

# Total recorded days per city
total_days_per_city = (
    city_day_clean
    .groupby('City')
    .size()
    .reset_index(name='Total_Days')
)

# Join and calculate percentage
good_air_pct = good_air_days.merge(total_days_per_city, on='City')
good_air_pct['Good_Air_%'] = (good_air_pct['Good_Days'] / good_air_pct['Total_Days'] * 100).round(1)
good_air_pct.sort_values('Good_Air_%', ascending=False, inplace=True)

print('📌 Cities ranked by % of days with Good AQI (≤ 50):')
display(good_air_pct)

### 6.10 Delhi Deep-Dive — Month-by-Month AQI Trend
How does Delhi's air quality vary throughout the year, and has it improved over time?

In [ ]:
# Filter to Delhi only
delhi_data = city_day_clean[city_day_clean['City'] == 'Delhi'].copy()

# Calculate monthly averages for each year
delhi_monthly = (
    delhi_data
    .groupby(['Year', 'Month'])[['AQI', 'PM2.5', 'PM10']]
    .mean()
    .round(1)
    .reset_index()
)

# Create a readable label like '2018-Nov'
delhi_monthly['Month_Name'] = delhi_monthly['Month'].map(month_names)
delhi_monthly['Period']     = delhi_monthly['Year'].astype(str) + '-' + delhi_monthly['Month_Name']
delhi_monthly.set_index('Period', inplace=True)

print('📌 Delhi monthly AQI, PM2.5 and PM10 over the years:')
display(delhi_monthly[['AQI', 'PM2.5', 'PM10']])

---
## Step 7: Export Cleaned Data (Optional)

Uncomment any line below to save the cleaned tables as CSV files.

In [ ]:
# ── Save individual cleaned files ────────────────────────────
# city_day_clean.to_csv('dataset/city_day_clean.csv',         index=False)
# city_hour_clean.to_csv('dataset/city_hour_clean.csv',       index=False)
# stations_clean.to_csv('dataset/stations_clean.csv',         index=False)
# station_day_clean.to_csv('dataset/station_day_clean.csv',   index=False)
# station_hour_clean.to_csv('dataset/station_hour_clean.csv', index=False)

# ── Save joined master tables ─────────────────────────────────
# station_day_master.to_csv('dataset/station_day_master.csv',   index=False)
# station_hour_master.to_csv('dataset/station_hour_master.csv', index=False)
# daily_master.to_csv('dataset/daily_master.csv',               index=False)
# hourly_master.to_csv('dataset/hourly_master.csv',             index=False)

print('Export cells ready. Uncomment a line above to save that file.')

---
## 📋 Final Summary

### ⭐ Top 5 Insights

| # | Insight | Key Number |
|---|---|---|
| 1 | **National AQI improved 47%** from 2015 to 2020 | 212.5 → 113.5 |
| 2 | **PM10 is the #1 AQI driver** (r = 0.80) — road dust dominates | Correlation 0.80 |
| 3 | **Winter is 2× worse than Monsoon** — Nov PM2.5 = 110 µg/m³ (3× safe limit) | Winter avg AQI 220 |
| 4 | **Delhi, Patna & Gurugram most polluted**; Aizawl is the cleanest city | Delhi avg AQI 259 |
| 5 | **COVID lockdown cut AQI by 42%** in Q2 2020 vs Q1 | 145 → 83 |

### Additional Findings
- Only **5.4%** of city-days have 'Good' air quality; **35.5%** are 'Moderate'
- **Gujarat and Delhi** are the worst states; **North-East India** is the cleanest region
- Air quality is **cleanest around 10:00** and worst at midnight (traffic + temperature inversion)
- **Patna** showed the biggest city-level improvement: −38% AQI from 2015 to 2019
- **NO2 and CO** spike in winter → dual contribution from vehicles and biomass/crop burning

### Cleaning Pipeline
| Step | Action |
|---|---|
| 1 | Removed duplicate rows |
| 2 | Dropped rows where all pollutants were missing |
| 3 | Capped physically impossible values → NaN |
| 4 | Clipped outliers at city/station 99th percentile |
| 5–7 | Filled gaps: interpolation → ffill/bfill → group median |
| 8 | Recalculated AQI using official CPCB sub-index formula |
| 9 | Re-derived AQI_Bucket from cleaned AQI using CPCB breakpoints |